# A quick introduction to PyTorch

PyTorch is a package for numerical computing in Python. Most things you can do in NumPy, you can also do in
PyTorch, with very similar syntax. If you're familiar with NumPy, this short introduction will mainly be to show you that basic operations in PyTorch are very similar to NumPy. 

Today, we're using PyTorch because: 
* It's an industry standard for deep learning because it augments the functionality of NumPy with 1) the ability to execute code on gpus, 2) built-in implementations of model classes commonly used in deep learning, 3) the ability to automatically perform certain kinds of optimization.
* We will be leveraging this automatic optimization to allow you to write your own model classes and loss functions, and have training (finding optimal parameters) happen automatically.

__CAUTION:__ If you mix NumPy or standard `math` library functions into your PyTorch code, PyTorch's automatic optimization will no longer work. Use PyTorch's built-in functions instead, e.g., `torch.exp` instead of `np.exp`


In [1]:
import torch
import numpy as np
# Most things you can do in NumPy, you can also do in PyTorch, with very similar syntax. For example:

a = np.arange(5) 
b = np.ones(5) * 2
c = np.exp(a) + b
print(f"Numpy {c=}")

at = torch.arange(5)
bt = torch.ones(5) * 2
ct = torch.exp(at) + bt
print(f"Torch {ct=}")

# The basic object in PyTorch is called a Tensor, which is very similar to a NumPy array. You can represent scalars,
# vectors, matrices, and matrices with more than two dimensions as Tensors.

print(f"A 1D Tensor (a vector): {torch.normal(0, 1, (3,))}")
print(f"A 2D Tensor (a matrix): {torch.normal(0, 1, (3, 2))}")
print(f"A 3D Tensor: {torch.normal(0, 1, (3, 2, 4))}")

# Element-wise operations: Like in NumPy, many functions in PyTorch operate element-wise, including basic operations
# * / + - ** between two tensors of the same shape, and things like torch.sqrt, torch.log

print(f"{torch.arange(3) * torch.arange(3)=}")
print(f"{torch.log(torch.arange(1, 5))=}")

# Broadcasting: Normally, for element-wise binary operations (e.g., * / + - **) to work between two tensors x and y,
# you need x.shape == y.shape. However, x and y can actually have different shapes but still allow such operations.
# Suppose x and y have different shapes, but for each dimension in which their shapes differ,
# either x or y has size one in that dimension. If this is truen then x and y are "compatible" for broadcasting.
# Like in NumPy, this means that when running any binary operations (e.g., * / + - **) on such tensors, PyTorch will
# repeat the smaller tensor multiple times along that dimension, to allow element-wise operations between them.
# This is called broadcasting. An example:

# Example
# A tensor with shape (2, 3)
a = torch.tensor([[1, 2, 3],
                  [4, 5, 6]])
# A tensor with shape (3,)
b = torch.tensor([[10, 20, 30]])
# The tensor `b` is broadcasted along dimension 0 to match the shape of `a`
c = a + b
print(c) # tensor([[11, 22, 33], [14, 25, 36]])



Numpy c=array([ 3.        ,  4.71828183,  9.3890561 , 22.08553692, 56.59815003])
Torch ct=tensor([ 3.0000,  4.7183,  9.3891, 22.0855, 56.5981])
A 1D Tensor (a vector): tensor([-0.5530,  0.6739,  1.4281])
A 2D Tensor (a matrix): tensor([[-1.0715,  0.2806],
        [-0.0402, -0.0893],
        [-0.7032,  0.1928]])
A 3D Tensor: tensor([[[-0.0764, -0.3229,  0.9377, -0.2711],
         [-0.9223, -0.4930,  0.7651, -0.6523]],

        [[ 0.2282,  0.9788, -0.9931,  0.5796],
         [ 0.7357,  0.1406,  0.7542,  1.4495]],

        [[ 0.2559, -0.5296, -2.0272,  0.4715],
         [ 0.4145, -1.5250, -1.3337,  1.0382]]])
torch.arange(3) * torch.arange(3)=tensor([0, 1, 4])
torch.log(torch.arange(1, 5))=tensor([0.0000, 0.6931, 1.0986, 1.3863])
tensor([[11, 22, 33],
        [14, 25, 36]])


In [2]:
# Some setup code for the assignment.
import pandas as pd
import numpy as np
from math import sqrt
import torch
from torch import Tensor
from torch.nn import Parameter
from torch.optim import LBFGS
from typing import Callable
from jaxtyping import Float


def optimize(
    f: Callable[[Float[Tensor, "p"], Float[Tensor, "n m"]], Float[Tensor, "n"]],
    L: Callable[[Float[Tensor, "n"]], Float[Tensor, "n"]],
    w: Float[Tensor, "p"],
    X: Float[Tensor, "n m"],
    y: Float[Tensor, "n"],
    max_iter=10,
) -> Float[Tensor, "p"]:
    """
    Optimize the parameters of a model using LBFGS.

    Parameters:
    - f: A function or model that takes features X and parameters w and returns predictions.
    - L: A loss function that takes predictions and labels y and returns a scalar loss.
    - w: A torch.Tensor containing the parameters to optimize.
    - X: A torch.Tensor containing the features.
    - y: A torch.Tensor containing the labels.

    Returns:
    - w: Optimized parameters.
    """

    w = Parameter(w)
    optimizer = torch.optim.LBFGS(
        [w],
        line_search_fn="strong_wolfe",
        max_iter=1000,
        tolerance_grad=1e-5,
        tolerance_change=1e-9,
    )

    # Define a function that computes the loss and
    # gradients of the loss with respect to model parameters
    def closure():
        optimizer.zero_grad()
        predictions = f(w, X)
        assert predictions.shape == y.shape, "Your predictions must be the same shape as y!"
        losses = L(predictions, y)
        assert predictions.shape == y.shape, "Your loss function must return values in the same shape as y!"
        loss = losses.mean()
        loss.backward()
        return loss

    # Run the optimizer
    for i in range(max_iter):
        optimizer.step(closure)

    return w.data



# Exercise 1: Model Classes and Loss Functions

So far in class we've seen a variety of __algorithms__, each defined by a choice of __model class__ and __loss function__. Here we'll implement a couple different algorithms in a hands-on way using __PyTorch__. 

PyTorch is a framework for numerical computation, similar to NumPy (in fact, much of the syntax is shared), but built for deep learning. We'll use it here for understanding linear models, and later in the course, for deep learning.

## Review: building blocks of supervised learning algorithms

Remember our procedure for finding a good model $f(x)$ of the relationship between features $x$ and label $y$.

1. We propose a __model class__, i.e. a set of possible models, e.g., lines. The model class has __parameters__ $w$, which allow you to select a model in the model class simply by selecting values for the parameters. 

2. We propose a __loss function__ $L(f_w(x), y)$, which takes in a prediction $f_w(x)$ and an observed label $y$ and tells you whether the prediction was close to the label.

3. We then find $\hat w$, the parameters corresponding to the best fit on the training data, by __optimizing__ the training loss: $\hat w = \arg \min_w \frac{1}{n}\sum_{i=1}^n L(f_w(x), y)$

4. Ultimately, we'll make predictions using the model $f_{\hat w}(x)$

## Exercise

In this exercise, you'll use PyTorch to implement a couple of model classes and loss functions. We've already written a function called `optimize` for you, which performs step 3. Suppose your model class takes $p$ parameters, and you have $n$ training observations, and $m$ features. `optimize` takes the following arguments:

* `f`: A function `f(w, X)`, which takes as arguments a $p$ Tensor of parameters `w` and an $n \times m$ Tensor of training data `X`
* `L`: A function `L(fx, y)` that takes an $n$ Tensor of predictions `fx`, a $n$ Tensor of labels `y`, and returns a scalar loss.
* `w`: A $p$ Tensor, containing a valid set of parameters. This serves to tell the model what shape the parameter should be, as well as to provide an initial guess for the optimal parameters.
* `X`: an $n \times m$ Tensor containing the features.
* `y`: an $n$ Tensor containing the labels.

To get you started, the next code block contains an implementation of a constant model with MSE loss, and optimizes it using the `optimize` function. 



In [3]:
# BEGIN EXAMPLE CODE
# -------------------------------------------------------------------------
def constant_model( 
    w: Float[Tensor, "p"], X: Float[Tensor, "n m"]
) -> Float[Tensor, "n"]:
    """
    A simple model that returns a constant value w regardless of x.
    Since we want to return one prediction per row of X, we need to repeat w n times.
    """
    n = X.shape[0]
    return w.repeat(n)


def squared_error_loss(
    fx: Float[Tensor, "n"],
    y: Float[Tensor, "n"]
) -> Float[Tensor, "n"]:
    return (y - fx) ** 2


# Example usage:
# Create fake dataset with 100 obs and 3 features
torch.manual_seed(0)
X = torch.normal(3, 1, (100, 3)) 
y = torch.normal(3, 1, (100,))

# Get predictions for X, for w = 1:
w = torch.ones(1)
preds = constant_model(w, X)
print(f"{preds=}")

# Evaluate training loss for w = 1:
train_loss = squared_error_loss(preds, y).mean()
print(f"Train loss at w=1: {train_loss}")

# Find the parameters that minimize the loss. Note that you are passing in *functions*
# `constant_model` and `squared_error_loss`. 
# `optimize` will call these repeatedly with different values of `w` to try and find the 
# best choice of `w`. 
# It will also make sure that you have defined your loss and model
# to return tensors with appropriate shapes.
w_hat = optimize(constant_model, squared_error_loss, w, X, y)

print(f"Optimal param: {w}")
print(f"Compare to the mean {y.mean()=}")
print("^^^These should be the same^^^")

# A more involved example: Linear regression
def linear_model(
    w: Float[Tensor, "p"], X: Float[Tensor, "n m"]
) -> Float[Tensor, "n"]:
    # This is the "right" way to write this, in matrix notation:
    fx = w[0] + X @ w[1:]
    
    # For your understanding: you could also write it like this,
    # using broadcasting:
    # fx = w[0] + torch.sum(w[1:] * X, dim=1)
    
    # Or even more explicitly:
    # n = X.shape[0]
    # fx = w[0].repeat(n) # intercept
    # for i in range(X.shape[1]):
    #     fx += w[i + 1] * X[:, i]  # Effect of i'th feature 

    return fx

# Parameter shape: intercept, and coefficient for each of 3 features.
linear_w = torch.ones(4)
linear_w_hat = optimize(linear_model, squared_error_loss, linear_w, X, y)
print("Optimal parameters for linear regression on this dataset:", linear_w_hat)

# END EXAMPLE CODE
# -------------------------------------------------------------------------



preds=tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])
Train loss at w=1: 6.490700721740723
Optimal param: tensor([3.2274])
Compare to the mean y.mean()=tensor(3.2274)
^^^These should be the same^^^
Optimal parameters for linear regression on this dataset: tensor([ 3.3585,  0.0134,  0.0535, -0.1059])


## Exercise 1.a: Implement Logistic Regression
In this exercise, similar to the provided example code, you will implement the __loss function__ and __model class__ for logistic regression, and call the optimize function to find the optimal solution on some training data.



In [4]:
# Fake dataset
torch.manual_seed(0)
logistic_y = torch.distributions.Bernoulli(0.3).sample((100,))


# 1.a.1 (1pt): Implement the logistic regression model
def logistic_regression_model(
    w: Float[Tensor, "p"], X: Float[Tensor, "n m"]
) -> Float[Tensor, "n"]:
    fx = w[0] + X @ w[1:]
    return torch.sigmoid(fx)


# 1.a.2 (2pt): Implement the logistic regression loss function 
def logistic_regression_loss(
    fx: Float[Tensor, "n"],
    y: Float[Tensor, "n"],
) -> Float[Tensor, "n"]:
    eps = 1e-18   # avoid log(0) error
    return -(y * torch.log(fx + eps) + (1 - y) * torch.log(1 - fx + eps))


    
logistic_w = torch.zeros(4)
logistic_w_hat = optimize(logistic_regression_model, logistic_regression_loss, logistic_w, X, logistic_y) # This should run
print(logistic_w_hat)

tensor([ 0.0088,  0.1763, -0.3327, -0.0624])


## Exercise 1.b: Adapting to a new type of data 

TransLink wants to predict the number of people arriving to get on the 99 bus at UBC every hour, based on the following features:

__Features__ `X`: 
* `X[:, 0]`: ${\bf 1}\{ \text{Is Raining} \}$
* `X[:, 1]`: ${\bf 1}\{ \text{Is Weekday} \}$
* `X[:, 2]`: ${\bf 1}\{ \text{Is Daytime} \}$

__Label__ `y`: Number of people arriving to the 99 bus stop

They collect this data at a granularity of one hour, and they have data from the past year.

They think that y is [Poisson distributed](https://en.wikipedia.org/wiki/Poisson_distribution) (a common assumption for count data), with a mean depending on x. They want you to build them a predictive model using [Poisson Regression](https://en.wikipedia.org/wiki/Poisson_regression).

Poisson Regression is a supervised learning algorithm, with model class

$$f_w(x) = \exp(w_0 + w_1 x_1 \ldots w_m x_m)$$

and it assumes that conditional on $x$, $y \sim \text{Poisson}(f_w(x))$. In other words :

$$p(y | x; w) = \text{Pois}(y ; f_w(x))$$

and chooses an optimal $w$ based on __Maximum Likelihood Estimation__. 

You have been provided with a function `poisson_likelihood` below that computes the function $p(y | x; w)$. This will be useful to you in computing the loss.



In [5]:
# Exercise 1b: Implement Poisson Regression

# Fake dataset
torch.manual_seed(0)
poisson_X = torch.distributions.Bernoulli(0.3).sample((100, 3)) 
poisson_y = torch.distributions.Poisson(1.).sample((100,))


def poisson_likelihood(
    fx: Float[Tensor, "n"],
    y: Float[Tensor, "n"],
):
    return torch.distributions.Poisson(fx).log_prob(y).exp()


# 1.b.1 (1pt): Implement the Poisson regression model 
def poisson_regression_model(
    w: Float[Tensor, "p"], X: Float[Tensor, "n m"]
) -> Float[Tensor, "n"]:
    fx = w[0] + X @ w[1:]
    return torch.exp(fx)


# 1.b.2 (2pt): Implement the Poisson regression loss function, using the `poisson_likelihood` function 
def poisson_regression_loss(
    fx: Float[Tensor, "n"],
    y: Float[Tensor, "n"],
) -> Float[Tensor, "n"]:
    return -poisson_likelihood(fx, y).log()


poisson_w = torch.zeros(4)
poisson_w_hat = optimize(poisson_regression_model, poisson_regression_loss, poisson_w, poisson_X, poisson_y)
print(poisson_w_hat)

tensor([ 0.0190, -0.0225,  0.1753, -0.1480])


### Exercise 1.b.3 (1pt): 

Q: What might be a problem with using a linear model class instead for this data?

A: A linear model can produce negative values with certain inputs which are infeasible for predicting the number of passengers.

# Exercise 2  (8 pts)

A mid- to high-end clothing brand wants to identify high-income customers from their demographic data, to better target their advertising campaigns. They've identified the population they'd like to market to as individuals with an income greater than \\$50,000.

They provide you with a census dataset with demographics similar to their potential marketing targets. Each observation is one individual, with features consisting of their demographic information, and a label of whether their income level is greater than \\$50,000. They've asked you to provide them with a predictive model that can identify which individuals have incomes greater than \\$50k, based on the individual's demographics.

In [6]:
# Setup code for exercise 2
import pandas as pd

census =  pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data", header=None)
census.columns = [
"age",
"workclass",
"fnlwgt",
"education",
"education-num",
"marital-status",
"occupation",
"relationship",
"race",
"sex",
"capital-gain",
"capital-loss",
"hours-per-week",
"native-country",
"income_level"
]

X = census.drop(columns=["income_level"])
y = census["income_level"] == " >50K"

## Exercise 2.a: Preprocessing

### Exercise 2.a.1
Use `train_test_split` to split X and y into training and testing DataFrames. Use 20% of your data for your test set, and include the argument `random_state=77` for reproducibility.

### Exercise 2.a.2 

Create a `ColumnTransformer` called `transformer` that does the following:

1. One-hot encodes all the categorical variables, dropping the first class
2. Scales all the numerical features using a `StandardScaler`


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=77)

 
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_columns = [
    "workclass",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country"
]

numerical_columns = [c for c in census.columns 
                     if c != 'income_level' and c not in categorical_columns]

transformer = ColumnTransformer(
    transformers=[
        ('categorical', OneHotEncoder(drop='first', handle_unknown='infrequent_if_exist'), categorical_columns),
        ('numerical', StandardScaler(), numerical_columns)
    ]
)





## Exercise 2b: Model Fitting
 
Fit a logistic regression model to your training data, using `sklearn.linear_model.LogisticRegression`. 
Use your column transformer to pre-process the data before fitting it. Assign your model to the variable `my_logit`.

In [8]:
from sklearn.linear_model import LogisticRegression

X_train_trans = transformer.fit_transform(X_train)
my_logit = LogisticRegression(random_state=77)
my_logit.fit(X_train_trans, y_train)

LogisticRegression(random_state=77)

## Exercise 2c: Prediction and Evaluation
The company wants you to output binary {0, 1} decisions. For any potential marketing target for whom you predict 0, they will not send an email. For those for whom you predict 1, they will. Since you're using logistic regression, you'll have to convert the model's predicted probabilities into a binary decision by choosing a threshold probability above which you output 1, and below which you output 0.

The cost of sending an email is \\$0.10. __Not counting the cost of the email__, the expected profit generated by a marketing email is \\$0.30 for individuals with income above \\$50,000, and \\$0.01 for individuals with income below that threshold. Assume that individuals you do not email will generate zero profit.

### Exercise 2.c.1 

Q: Based on these costs, what decision threshold (i.e., probability threshold) should you choose for your logistic regression classifier ? Assign it to the `decision_threshold` variable below. Show your reasoning for partial credit.

A: If an email is sent, the net profit for each high-income user is \\$0.30 - \\$0.10 = \\$0.20, while the net profit for each low-income user is \\$0.01 - \\$0.10 = -\\$0.09. If an email is not sent, the net profit is \\$0. Let p be the decision threshold. To achieve positive profit, p should be set such that the expected profit of emailing is greater than that of not emailing, i.e. 0.2 * p + (1 - p) * (-0.09) > 0. Solve for p, we can obtain p > (0.09 / 0.29) = 0.3103....


In [1]:
# Exercise 2.c.1
decision_threshold = 0.09 / 0.29 # These should be floats



### Exercise 2.c.2

What profit would the firm realize on the test set, if they act according to your predictions at the threshold you set? Assign your answer to the `test_profit` variable below.

Hint: You may need to use the `.predict_proba` method of the LogisticRegression class. You will recieve credit for this as long as your answer is consistent with the `decision_threshold` you set, even if that was wrong.
 

In [10]:
X_test_trans = transformer.transform(X_test)
y_pred = my_logit.predict_proba(X_test_trans)

test_profits = []
for p_pred_high_income, is_actual_high_income in zip(y_pred[:, 1], y_test):
    if p_pred_high_income > decision_threshold:
        if is_actual_high_income == 1:
            test_profits.append(0.2)
        else:
            test_profits.append(-0.09)
    else:
        test_profits.append(0)

test_profit = sum(test_profits)
print(test_profit)

179.24999999999602


/home/jl/miniconda3/envs/bait580_2024/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [11]:
# Test code: you can run this as a basic check as to whether you've filled out each field enough for the autograder to run properly.
# Be warned, it's not comprehensive (just checks types and shapes) and doesn't tell you whether you completed the task correctly!
# While not required, it's often good practice for you to write your own test cases (in addition to the ones provided here) to check that your functions behave as you would expect. 

def isa_scalar(x):
    return (
        isinstance(x, float)
        or (
            isinstance(x, Tensor)
            and x.view(-1).shape == (1,)
        )
        or (
            isinstance(x, np.ndarray)
            and x.reshape(-1).shape == (1,)
        )  
    )

def check_scalar(name, x):
    assert isa_scalar(x), f"{name} has value {x}; should be a scalar"

def check_class(name, x, cls):
    assert isinstance(x, cls), f"{name} has type {type(x)}; should be {cls}"



# Exercise 1
_X = torch.normal(3, 1, (100, 3)) 

assert logistic_regression_model(torch.ones(4), _X).shape == (_X.shape[0],)
assert logistic_regression_loss(torch.ones(4), torch.ones(4)).shape == (4,)
    
assert poisson_regression_model(torch.ones(4), _X).shape == (_X.shape[0], )
assert poisson_regression_loss(torch.ones(4), torch.ones(4)).shape == (4,)

# Exercise 2    
check_class("transformer", transformer, ColumnTransformer)
check_class("my_logit", my_logit, LogisticRegression)
check_scalar("decision_threshold", decision_threshold)
check_scalar("test_profit", test_profit)

print("Tests passed!")

Tests passed!
